# Colab 3: Combined Two-Dataset Training

This notebook mounts Google Drive, clones the project from GitHub, auto-discovers the dataset folders in Drive, trains a combined model, evaluates it, zips the outputs, and downloads the zip.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/au510621104021/FND_2027.git'
BRANCH = 'main'
PROJECT_DIR = '/content/FND_2027'

# Drive folder to scan for datasets. Keep this broad unless your datasets are elsewhere.
DRIVE_SEARCH_ROOT = '/content/drive/MyDrive'

# Optional: folder in Drive where the final zip will be copied.
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/FND_2027_outputs'

# Dataset folder name hints used for auto-discovery.
DATASET_1_HINTS = ['ISOT Fake News Dataset', 'ISOT']
DATASET_2_HINTS = ['Fake News Dataset']

# If auto-discovery finds the wrong folders, hardcode them here and rerun from this cell.
DATASET_DIR_1 = None
DATASET_DIR_2 = None

In [ ]:
import os
from pathlib import Path

os.chdir('/content')
!rm -rf /content/FND_2027
!git clone --branch main https://github.com/au510621104021/FND_2027.git /content/FND_2027

assert Path(PROJECT_DIR).exists(), f'Project directory not found: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('Project directory:', os.getcwd())
!ls

In [ ]:
import os
from pathlib import Path

os.chdir(PROJECT_DIR)
assert Path('requirements.txt').exists(), 'requirements.txt not found. Clone step likely failed.'
!pip install -q -r requirements.txt

In [ ]:
import os
from pathlib import Path

search_root = Path(DRIVE_SEARCH_ROOT)
assert search_root.exists(), f'Drive search root not found: {DRIVE_SEARCH_ROOT}'

def find_dataset_dir(hints):
    candidates = []
    for path in search_root.rglob('*'):
        if not path.is_dir():
            continue
        name_lower = path.name.lower()
        if any(h.lower() in name_lower for h in hints):
            score = 0
            for h in hints:
                if path.name.lower() == h.lower():
                    score += 10
                elif h.lower() in name_lower:
                    score += 3
            candidates.append((score, len(path.parts), str(path)))
    if not candidates:
        return None
    candidates.sort(key=lambda x: (-x[0], x[1], x[2]))
    return candidates[0][2]

if DATASET_DIR_1 is None:
    DATASET_DIR_1 = find_dataset_dir(DATASET_1_HINTS)
if DATASET_DIR_2 is None:
    DATASET_DIR_2 = find_dataset_dir(DATASET_2_HINTS)

print('Detected DATASET_DIR_1 =', DATASET_DIR_1)
print('Detected DATASET_DIR_2 =', DATASET_DIR_2)

assert DATASET_DIR_1 and Path(DATASET_DIR_1).exists(), (
    'Could not auto-find dataset 1. Update DRIVE_SEARCH_ROOT or hardcode DATASET_DIR_1.'
)
assert DATASET_DIR_2 and Path(DATASET_DIR_2).exists(), (
    'Could not auto-find dataset 2. Update DRIVE_SEARCH_ROOT or hardcode DATASET_DIR_2.'
)

print('\nDataset 1 files:')
!ls "$DATASET_DIR_1" | head
print('\nDataset 2 files:')
!ls "$DATASET_DIR_2" | head

In [ ]:
import os
import yaml
from pathlib import Path

os.chdir(PROJECT_DIR)
config_path = Path('config/config.yaml')
with config_path.open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

config['data']['dataset_name'] = 'generic'
config['data']['data_dir'] = DATASET_DIR_1
config['data']['dataset_names'] = ['generic', 'generic']
config['data']['data_dirs'] = [DATASET_DIR_1, DATASET_DIR_2]
config['logging']['checkpoint_dir'] = './checkpoints'
config['logging']['log_dir'] = './logs'
config['inference']['model_checkpoint'] = './checkpoints/best_model_combined.pt'

with config_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, sort_keys=False)

print('Updated config for combined training:')
print(config['data'])

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python scripts/train.py --config config/config.yaml

In [ ]:
import os
import shutil
from pathlib import Path

os.chdir(PROJECT_DIR)
best_model = Path('checkpoints/best_model.pt')
combined_model = Path('checkpoints/best_model_combined.pt')
assert best_model.exists(), 'Training did not produce checkpoints/best_model.pt'
shutil.copy2(best_model, combined_model)
print(f'Saved combined checkpoint to: {combined_model}')

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python scripts/evaluate.py --config config/config.yaml --checkpoint checkpoints/best_model_combined.pt

In [ ]:
import os
import shutil
from pathlib import Path

os.chdir(PROJECT_DIR)
zip_base = Path('combined_model_artifacts')
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir='.', base_dir='checkpoints')
print(f'Created zip: {zip_path}')
!zip -ur combined_model_artifacts.zip results logs -x "*.ipynb_checkpoints*"

In [ ]:
import os
from pathlib import Path
from google.colab import files

os.chdir(PROJECT_DIR)
zip_path = Path('combined_model_artifacts.zip')
assert zip_path.exists(), 'Zip file was not created.'

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
!cp combined_model_artifacts.zip "$DRIVE_OUTPUT_DIR/"
print(f'Copied zip to Drive: {DRIVE_OUTPUT_DIR}')

files.download(str(zip_path))